# exp057 — exp045 en versión eficiente (entrada 224, 3 slots, sin TTA) en `split_v1_seed42` fold 0

Padre: exp045_raptor_aug_espacial. Experimento de PRODUCTO (premio de eficiencia): agrupa a propósito los cambios de coste
(entrada 320 → 224 por interpolación bilineal, solo slots 0–2 = Sag fluido, Sag, Cor fluido; la inferencia ya no usa TTA)
porque la pregunta es el trade-off AUC vs runtime, no aislar una variable. Ver SPEC.md.

# exp045 — exp031 + augmentation espacial (espejo L/R anatómico p 0.5 + afín por estudio) en `split_v1_seed42` fold 0

Padre: exp031_raptor_etiquetas_consenso. Única variable: augmentation espacial de entrenamiento en GPU: espejo L/R
anatómico (mismo eje que el TTA `_mirror_tri` de nartaa: sagital → invertir los 3 cortes-canal; coronal/axial → volteo
del ancho) con p 0.5 por estudio + afín por estudio (rot ±10°, escala 0.9–1.1, traslación ±6 %). Todo lo demás idéntico.

Notas del padre (exp031):

# exp031 — Receta Raptor (exp029) con etiquetas de entrenamiento de consenso (Raptor + L2 + L3 + L4) en `split_v1_seed42` fold 0

Padre: exp029_raptor_entreno. Única variable: etiquetas de ENTRENAMIENTO = promedio por celda de 4 fuentes blandas
(Raptor `labels_llm_soft`, L2 stevenleehans v4_blend, L3 pilkwang report_labels_v2, L4 lixin73 gpt56sol); si falta
una fuente en una celda se promedian las presentes. Todo lo demás idéntico al padre.

Notas del padre:


SPEC: `SPEC.md` en esta carpeta. Porta a celdas el script público `dreaddevelopment/knee-mri-training-the-twelve-finding-model`
(CoAtNet-2 384 + atención MIL por hallazgo, 2.5D) sobre el corpus denso 96×384 construido con `build_corpus96.py`
(script auxiliar de esta carpeta = `build_study` verbatim de `nartaa/rsna-knee-0949-anatomical-mirror`).
Cambios contra el script público (declarados en la SPEC): selección de época y SWA por **pseudo AUC L2 en val fold 0**
(nunca gold), split `split_v1_seed42`, entrada = recorte central 320 del corte de 384 (como la inferencia de nartaa).

Celdas fijas: **0** parámetros · **1** setup · **2** datos y split · **3** modelo · **4** entrenamiento · **5** evaluación ·
**6** guardar · **7** verdict.

## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp078_seqhead_bigru"
PADRE = "exp074"
FOLDS = [0]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True
ALL_DATA = False          # True: entrena con los 4349 no-gold, sin validación; SWA sobre SWA_EPOCHS fijas
SWA_EPOCHS = []           # solo ALL_DATA: épocas (0-index) a promediar, fijadas con lo aprendido en fold 0

CORPUS = "/workspace/data/cache/raptor96_384"   # seqhead: el mismo corpus (build_corpus96.py) leído del volumen; solo se lee una vez para los embeddings
LABELS_SOFT = "/workspace/data/labels/public/dreaddevelopment/rsna-knee-labels/labels_llm_soft.parquet"  # fuente 1 (la del padre)
L3_CSV = "/workspace/data/labels/public/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"  # exp031: fuente 3
L4_CSV = "/workspace/data/labels/public/lixin73/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv"  # exp031: fuente 4
L2_CSV = "/workspace/data/labels/public/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"  # vara y fuente 2
V1_CSV = "/workspace/data/labels/report_labels_v1.csv"
TEACHER_CSV = "/workspace/users/rodrigo/wt_exp074_raptor224_distill/results/exp074_raptor224_distill/teacher.csv"  # exp074: media en prob. 0.5/0.5 R7 compact + nartaa 384 (fuera de git)
TEACHER_W = 0.5          # exp074: objetivo = TEACHER_W × teacher + (1 − TEACHER_W) × consenso exp031
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
IMG = 384                 # lado guardado en el corpus
RES = 320                 # recorte central RES del corte IMG (como la inferencia nartaa)
RES_IN = 224              # exp057: el recorte RES se reescala a RES_IN (bilineal, align_corners=False, como el kernel Ch)
USE_SLOTS = [0, 1, 2]     # exp057: solo Sag fluido, Sag, Cor fluido (cortes 0–65); sin Cor no-fluido ni Axial
BENCH_N = 64              # exp057: estudios para medir inferencia por estudio (exp057 vs configuración exp045)
NORM = "imagenet"
K = 12
K_EVAL = 24
EPOCHS = 16
BATCH_SIZE = 8
BB_LR = 3e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.02
PCT_START = 0.15
GRAD_CLIP = 3.0
TOPK = 3                  # SWA = media de las TOPK mejores épocas por pseudo L2
L2_THRESHOLD = 0.5
GRAD_CKPT = True
AUG_FLIP_P = 0.5          # exp045: espejo L/R anatómico por estudio (padre: sin espejo)
AUG_ROT_DEG = 10.0        # exp045: afín por estudio, rotación U(-10°, 10°)
AUG_SCALE = [0.9, 1.1]    # exp045: escala U(0.9, 1.1)
AUG_TRANS = 0.06          # exp045: traslación U(-6 %, 6 %) del lado, en x e y
N_WORKERS = 12
LOG_EVERY = 50
NOTES = "seqhead: backbone exp074 swa_f0 congelado, embeddings de 64 ventanas ordenadas; cabeza Bi-GRU (2 x 256) + attention-MIL; control = cabeza del padre reentrenada con el mismo protocolo | padre exp074: objetivo 0.5 teacher + 0.5 consenso, Raptor 224 slots 0-2"
# seqhead: backbone CoAtNet del padre CONGELADO; embeddings de ventana una vez; solo se entrena la cabeza (BB_LR y AUG_* no se usan)
HEAD = "bigru"              # cabeza nueva: "gru" (exp077) | "bigru" (exp078); el control "mil" = cabeza del padre
BACKBONE_CKPT = "/workspace/users/rodrigo/wt_exp074_raptor224_distill/results/exp074_raptor224_distill/swa_f0.pt"   # exp074 fold 0 (folds 1-4), SWA ép 14,15,13
BACKBONE_CKPT_ALLDATA = "/workspace/users/rodrigo/kaggle_export/exp074_raptor224_distill/swa_alldata.pt"            # solo ALL_DATA
EMB_DIR = "/root/seqhead_emb"   # caché de embeddings por sha del checkpoint (fuera de git)
K_SEQ = 64                 # ventanas por estudio (= K_EVAL del kernel exp074-student224), ordenadas por corte
GRU_HIDDEN = 256           # por sentido
EMB_BATCH = 256            # ventanas por forward del backbone (4 estudios)
INFO_SEEDS = [7, 123]      # semillas extra de control y cabeza nueva: solo informativas, no deciden
CRIT_WORKED = 0.9037       # SPEC: SWA pseudo f0 >= padre 0.8987 + 0.005 y > control
CRIT_KILLED = 0.8997       # SPEC: SWA gold f0 < padre 0.9197 - 0.02

## 1. Setup

In [ ]:
import copy
import json
import math
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import decision, git_info, guardar_metricas, leer_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona checkpoints (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
assert not ALL_DATA or len(FOLDS) == 1
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
if SMOKE:
    EPOCHS = 1
TAG = "alldata" if ALL_DATA else "f"
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · "
      f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'} · timm {timm.__version__} · torch {torch.__version__}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (las revisiones detienen el notebook si algo está mal)

In [ ]:
def _read(path, cols=None):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    return df.set_index("StudyInstanceUID")[cols or LABELS].astype(float)

split = load_split(SPLIT)
def _read_soft(path, name):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path, low_memory=False)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    assert not df.StudyInstanceUID.duplicated().any(), f"{name} con StudyInstanceUID duplicados"
    df = df.set_index("StudyInstanceUID")[LABELS].apply(pd.to_numeric, errors="coerce").astype(float)
    assert df.stack().between(0.0, 1.0).all(), f"{name} fuera de [0, 1]"
    return df

# exp031: etiquetas de ENTRENAMIENTO = promedio por celda de 4 fuentes blandas (si falta una fuente, media de las presentes)
SOURCES = {"raptor": _read_soft(LABELS_SOFT, "raptor"), "L2": _read_soft(L2_CSV, "L2"),
           "L3": _read_soft(L3_CSV, "L3"), "L4": _read_soft(L4_CSV, "L4")}
_all = sorted(set().union(*[s.index for s in SOURCES.values()]))
_stack = np.stack([s.reindex(_all)[LABELS].to_numpy(dtype=float) for s in SOURCES.values()])
_n_src = np.isfinite(_stack).sum(0)
soft = pd.DataFrame(np.where(_n_src > 0, np.nansum(_stack, 0) / np.maximum(_n_src, 1), np.nan),
                    index=pd.Index(_all, name="StudyInstanceUID"), columns=LABELS)
N_SRC = pd.DataFrame(_n_src, index=soft.index, columns=LABELS)
l2 = _read(L2_CSV)                    # vara interna: L2 binarizada > 0.5
v1 = _read(V1_CSV)                    # informativo
train_csv = pd.read_csv("/workspace/data/raw/train.csv")
train_csv["StudyInstanceUID"] = train_csv.StudyInstanceUID.astype(str)
gold = train_csv.dropna(subset=LABELS, how="all").set_index("StudyInstanceUID")[LABELS].astype(float)
gold_ids = list(gold.index)
assert len(gold) == 58 and set(gold_ids) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
soft = soft.dropna(how="all")         # estudio sin ninguna fuente fuera; celda sin fuentes (NaN) se enmascara en la pérdida
assert np.nanmin(soft.values) >= 0 and np.nanmax(soft.values) <= 1
print(f"etiquetas de entrenamiento (consenso): {len(soft)} estudios · celdas NaN {int(soft.isna().sum().sum())}")

# exp074: destilación. Fila del teacher con NaN = estudio que el teacher no puntuó -> se queda el consenso solo.
teacher = pd.read_csv(TEACHER_CSV)
teacher["StudyInstanceUID"] = teacher.StudyInstanceUID.astype(str)
assert not teacher.StudyInstanceUID.duplicated().any()
teacher = teacher.set_index("StudyInstanceUID")[LABELS].astype(float)
_tv = teacher.to_numpy()[np.isfinite(teacher.to_numpy())]
assert ((_tv >= 0) & (_tv <= 1)).all()
_ng = [u for u in soft.index if u not in set(gold_ids)]
_miss = [u for u in _ng if u not in teacher.index]
assert not _miss, f"teacher sin {len(_miss)} estudios de train"
_t = teacher.reindex(soft.index)
_tok = _t.notna().all(axis=1)
# R7 deja sin puntuar algunos estudios grandes (fallback de su worker, igual que en Kaggle): ahí queda el consenso solo
assert (~_tok.loc[_ng]).mean() < 0.10, f"teacher falló en {int((~_tok.loc[_ng]).sum())} estudios"
SOFT_CONSENSO = soft.copy()
_m = _tok.to_numpy()[:, None]
soft = pd.DataFrame(np.where(_m, TEACHER_W * _t.to_numpy() + (1 - TEACHER_W) * soft.to_numpy(), soft.to_numpy()),
                    index=soft.index, columns=LABELS)
print(f"teacher: {len(teacher)} estudios · sin puntuar en train {int((~_tok.loc[_ng]).sum())} · "
      f"media teacher {np.nanmean(_t.loc[_ng].to_numpy()):.3f} · consenso {np.nanmean(SOFT_CONSENSO.loc[_ng].to_numpy()):.3f} · "
      f"objetivo {np.nanmean(soft.loc[_ng].to_numpy()):.3f}")
print("acuerdo binarizado teacher–consenso (> 0.5) por hallazgo: " + " · ".join(
    f"{c} {np.nanmean((_t.loc[_ng, c] > 0.5) == (SOFT_CONSENSO.loc[_ng, c] > 0.5)):.3f}" for c in LABELS))

corpus_ids = np.load(f"{CORPUS}/ids.npy").astype(str)
corpus_masks = np.load(f"{CORPUS}/masks.npy")
ID2ROW = {u: i for i, u in enumerate(corpus_ids)}
D = corpus_masks.shape[1]
assert D == 96, D
meta = json.load(open(f"{CORPUS}/build_meta.json"))
print(f"corpus {len(corpus_ids)} × {D} · cortes válidos medios {corpus_masks.sum(1).mean():.1f} · "
      f"estudios vacíos {(corpus_masks.sum(1) == 0).sum()} · errores build {len(meta['errors'])}")
assert set(gold_ids) <= set(ID2ROW)

pseudo_bin = (l2 > L2_THRESHOLD).astype(float).where(l2.notna())
v1_bin = v1.where(v1.isin([0.0, 1.0]))

def fold_ids(fold):
    """train / val para un fold de split_v1; train = pool sin el fold ∩ etiquetas de entrenamiento (consenso)."""
    if ALL_DATA:
        tr = [u for u in soft.index if u not in set(gold_ids) and u in ID2ROW]
        return tr, []
    pool = split[split.fold >= 0]
    tr = [u for u in pool.loc[pool.fold != fold, "StudyInstanceUID"] if u in soft.index and u in ID2ROW]
    va = [u for u in pool.loc[pool.fold == fold, "StudyInstanceUID"] if u in ID2ROW]
    return tr, va

for f in FOLDS:
    tr, va = fold_ids(f)
    assert not set(tr) & set(gold_ids), "gold dentro del entrenamiento"
    assert not set(tr) & set(va)
    print(f"fold {f}: train {len(tr)} · val {len(va)} (L2 cubre {pseudo_bin.index.isin(va).sum()}) · gold {len(gold_ids)}")
    _ns = N_SRC.reindex(tr).fillna(0).to_numpy()
    print(f"  celdas de train por nº de fuentes (0..4): {[int((_ns == k).sum()) for k in range(5)]} · "
          f"cobertura por fuente (estudios de train): {({n: int(pd.Index(tr).isin(s.index).sum()) for n, s in SOURCES.items()})}")
    _tr = {n: s.reindex(tr)[LABELS].to_numpy() for n, s in SOURCES.items()}
    _agree = lambda a, b: np.mean([(x > 0.5) == (y > 0.5) for x, y in zip(_tr[a].ravel(), _tr[b].ravel())
                                   if np.isfinite(x) and np.isfinite(y)])
    print("  acuerdo binarizado (> 0.5) en train: " + " · ".join(f"{a}–{b} {_agree(a, b):.3f}" for a, b in
          [("L2", "L4"), ("L2", "L3"), ("L3", "L4"), ("raptor", "L2"), ("raptor", "L3"), ("raptor", "L4")]))
    print(f"  media del objetivo en train: consenso {np.nanmean(soft.loc[tr].to_numpy()):.3f} · " +
          " · ".join(f"{n} {np.nanmean(v):.3f}" for n, v in _tr.items()))

## 3. Modelo (verbatim del script público: `build_backbone` + `RaptorClassifier`; dataset `StudyWindows`)

In [ ]:
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
SLOT_SIZES = [26, 22, 18, 12, 18]     # exp045: slots del corpus (Sag fluid, Sag, Cor fluid, Cor, Ax), como el TTA nartaa
SAGITTAL_SLOTS = (0, 1)
SLOT_OF = np.repeat(np.arange(len(SLOT_SIZES)), SLOT_SIZES)
assert len(SLOT_OF) == D
IN_SLOTS = np.isin(SLOT_OF, USE_SLOTS)      # exp057: cortes de los slots usados


class StudyWindows(Dataset):
    """Bolsa de ventanas 2.5D por estudio (3 cortes consecutivos -> RGB) desde el corpus memmap.
    Igual que el script público salvo: recorte central RES del corte IMG (en vez de resize) y la conversión a float,
    la ganancia ×U(0.9,1.1) y la normalización ImageNet se hacen en GPU (mismas operaciones)."""
    def __init__(self, ids, k, train):
        self.ids, self.k, self.train = list(ids), k, train
        self.vols = None

    def __len__(self):
        return len(self.ids)

    def _centers(self, valid, count):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, D - 2))]
        if self.train:
            pool = cs * (count // len(cs) + 1)
            random.shuffle(pool)
            return pool[:count]
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return [cs[i] for i in idx]

    def __getitem__(self, i):
        if self.vols is None:
            self.vols = np.load(f"{CORPUS}/vols.npy", mmap_mode="r")
        uid = self.ids[i]; row = ID2ROW[uid]
        valid = np.where((corpus_masks[row] > 0) & IN_SLOTS)[0]     # exp057: solo slots USE_SLOTS
        if len(valid) < 3:
            valid = np.arange(min(3, D))
        cs = np.array([max(1, min(c, D - 2)) for c in self._centers(valid, self.k)])
        lo = (IMG - RES) // 2
        vol = self.vols[row]
        tri = np.stack([vol[cs - 1, lo:lo + RES, lo:lo + RES], vol[cs, lo:lo + RES, lo:lo + RES],
                        vol[cs + 1, lo:lo + RES, lo:lo + RES]], 1)          # (K,3,RES,RES) uint8
        gain = 1.0 + (random.random() - 0.5) * 0.20 if self.train else 1.0
        sag = torch.from_numpy(np.isin(SLOT_OF[cs], SAGITTAL_SLOTS))     # exp045: eje del espejo por ventana (slot del corte central)
        return torch.from_numpy(np.ascontiguousarray(tri)), torch.tensor(gain, dtype=torch.float32), i, sag


def spatial_aug(x, sag):
    """exp045: espejo L/R anatómico (p AUG_FLIP_P) + afín (misma matriz para las K ventanas de un estudio), en GPU.
    Espejo = `_mirror_tri` del TTA nartaa: ventana con corte central sagital -> invertir los 3 cortes-canal; si no -> volteo W."""
    B, K_, C, H, W = x.shape
    flip = (torch.rand(B, device=x.device) < AUG_FLIP_P).view(B, 1, 1, 1, 1)
    sag = sag.to(x.device).view(B, K_, 1, 1, 1)
    x = torch.where(flip, torch.where(sag, x.flip(2), x.flip(-1)), x)
    ang = (torch.rand(B, device=x.device) * 2 - 1) * math.radians(AUG_ROT_DEG)
    scl = AUG_SCALE[0] + torch.rand(B, device=x.device) * (AUG_SCALE[1] - AUG_SCALE[0])
    tr = (torch.rand(B, 2, device=x.device) * 2 - 1) * AUG_TRANS * 2      # coords normalizadas [-1, 1]: lado = 2
    cos, sin = torch.cos(ang) / scl, torch.sin(ang) / scl                  # theta: salida -> entrada (escala > 1 = zoom in)
    theta = torch.stack([torch.stack([cos, -sin, tr[:, 0]], 1), torch.stack([sin, cos, tr[:, 1]], 1)], 1)  # (B,2,3)
    theta = theta.repeat_interleave(K_, 0)
    flat = x.reshape(B * K_, C, H, W)
    grid = F.affine_grid(theta, flat.shape, align_corners=False)
    return F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False).view(B, K_, C, H, W)


def to_input(x_u8, gain, sag=None, train=False):
    x = x_u8.to(DEVICE, non_blocking=True).float() / 255.0
    if train:
        x = spatial_aug(x, sag)                                            # exp045: solo entrenamiento
    if RES_IN != x.shape[-1]:                                              # exp057: 320 -> 224 antes de normalizar
        B_, K__ = x.shape[:2]
        x = F.interpolate(x.flatten(0, 1), size=(RES_IN, RES_IN), mode="bilinear", align_corners=False).view(B_, K__, 3, RES_IN, RES_IN)
    if not torch.all(gain == 1.0):
        x = (x * gain.to(DEVICE).view(-1, 1, 1, 1, 1)).clamp(0, 1)
    if NORM == "imagenet":
        x = (x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)
    return x


def build_backbone(arch, pretrained=True):
    return timm.create_model(arch, pretrained=pretrained, num_classes=0, in_chans=3, global_pool="avg", img_size=RES_IN)   # exp057


class RaptorClassifier(nn.Module):
    """Raptor encoder + per-diagnosis attention-MIL head (12 findings)."""
    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K_ = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).view(B, K_, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.encode(x))


def build_model():
    bb = build_backbone(ARCH, pretrained=True)
    if GRAD_CKPT:
        bb.set_grad_checkpointing(True)
    return RaptorClassifier(bb, F_dim=bb.num_features).to(DEVICE)



# ---- seqhead: backbone del padre CONGELADO -> embeddings de ventana (una vez, en caché) -> solo se entrena la cabeza ----
import hashlib


def _sha(path, n=12):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()[:n]


def load_frozen(ckpt_path):
    """Modelo del padre (backbone + cabeza) con el checkpoint dado, congelado (eval, sin gradiente)."""
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    bb = build_backbone(ARCH, pretrained=False)
    m = RaptorClassifier(bb, F_dim=bb.num_features)
    m.load_state_dict(ck["model"], strict=True)
    for p in m.parameters():
        p.requires_grad_(False)
    return m.to(DEVICE).eval(), ck


@torch.no_grad()
def embed_windows(ckpt_path, ids):
    """(len(ids), K_SEQ, F) float16 = backbone congelado sobre las K_SEQ ventanas de evaluación de cada estudio
    (StudyWindows train=False: linspace sobre los centros válidos de USE_SLOTS, ordenadas por corte; = ventanas del kernel).
    Sin aug. Caché en EMB_DIR por sha del checkpoint: solo se calculan los estudios que faltan."""
    tag = f"{_sha(ckpt_path)}_k{K_SEQ}_r{RES_IN}"
    d = Path(EMB_DIR)
    d.mkdir(parents=True, exist_ok=True)
    fe, fd = d / f"emb_{tag}.npy", d / f"done_{tag}.npy"
    model, _ = load_frozen(ckpt_path)
    F_ = model.backbone.num_features
    mm = np.lib.format.open_memmap(fe, mode="r+" if fe.exists() else "w+", dtype=np.float16,
                                   shape=(len(corpus_ids), K_SEQ, F_))
    done = np.load(fd) if fd.exists() else np.zeros(len(corpus_ids), bool)
    todo = [u for u in ids if not done[ID2ROW[u]]]
    t0 = time.time()
    if todo:
        dl = DataLoader(StudyWindows(todo, K_SEQ, False), batch_size=EMB_BATCH // K_SEQ, shuffle=False,
                        num_workers=N_WORKERS, pin_memory=True)
        n, nxt = 0, 500
        for x, g, idx, _sag in dl:
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                f = model.encode(to_input(x, g)).float()
            rows = [ID2ROW[todo[i]] for i in idx.tolist()]
            mm[rows] = f.cpu().numpy().astype(np.float16)
            done[rows] = True
            n += len(rows)
            if n >= nxt:
                nxt += 500
                print(f"  embeddings {n}/{len(todo)} · {(time.time() - t0) / n:.3f} s/estudio", flush=True)
        mm.flush()
        np.save(fd, done)
    secs = time.time() - t0
    print(f"embeddings {tag}: {len(ids)} estudios ({len(todo)} nuevos, {secs:.0f} s) · F {F_}", flush=True)
    out = torch.from_numpy(np.ascontiguousarray(mm[[ID2ROW[u] for u in ids]]))
    del model
    torch.cuda.empty_cache()
    return out, tag, secs


class MILHead(nn.Module):
    """Control: la cabeza del padre (`RaptorClassifier.head`) sin backbone; mismos nombres de parámetros."""
    def __init__(self, F_dim, n=12, drop=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)

    def forward(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb


class SeqGRUHead(nn.Module):
    """seqhead: LayerNorm -> GRU (uni o bi) sobre las ventanas ordenadas por corte -> LayerNorm -> attention-MIL por
    hallazgo (misma atención y clasificador que el padre, sobre los estados de la GRU)."""
    def __init__(self, F_dim, n=12, hidden=256, bidirectional=False, drop=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(F_dim)
        self.gru = nn.GRU(F_dim, hidden, num_layers=1, batch_first=True, bidirectional=bidirectional)
        H = hidden * (2 if bidirectional else 1)
        self.norm2 = nn.LayerNorm(H)
        self.att = nn.Sequential(nn.Linear(H, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, H))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)

    def forward(self, feats):
        s, _ = self.gru(self.norm(feats))
        s = self.norm2(s)
        a = torch.softmax(self.att(s), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, s)
        return (pooled * self.clsW).sum(-1) + self.clsb


def make_head(kind, F_dim):
    if kind == "mil":
        return MILHead(F_dim).to(DEVICE)
    assert kind in ("gru", "bigru"), kind
    return SeqGRUHead(F_dim, hidden=GRU_HIDDEN, bidirectional=kind == "bigru").to(DEVICE)

## 4. Entrenamiento (bucle del script público; selección y top-K por pseudo L2 en val, nunca gold)

In [ ]:
def score(pred):
    r = {}
    for name, ref in (("pseudo", pseudo_bin), ("v1", v1_bin), ("gold", gold)):
        ids = [u for u in pred.index if u in ref.index]
        r[name] = mean_auc(ref.loc[ids, LABELS].to_numpy(), pred.loc[ids, LABELS].to_numpy()) if ids else None
    return r


def average_states(states):
    avg = {}
    for k_ in states[0]:
        v0 = states[0][k_]
        avg[k_] = (sum(s[k_].double() for s in states) / len(states)).to(v0.dtype) if v0.is_floating_point() else v0.clone()
    return avg


def save_preds(pred_val, pred_gold, name):
    if len(pred_val):
        pred_val.add_prefix("pred_").reset_index().to_csv(RESULTS / f"oof_{name}.csv", index=False)
    pred_gold.add_prefix("pred_").reset_index().to_csv(RESULTS / f"gold_{name}.csv", index=False)


def run_ids(fold):
    tr, va = fold_ids(fold)
    return (tr[:64], va[:48]) if SMOKE else (tr, va)


# seqhead: embeddings de todos los estudios que se usan (train + val + gold), una sola vez, con el backbone congelado
T_CELL4 = time.time()
CKPT = BACKBONE_CKPT_ALLDATA if ALL_DATA else BACKBONE_CKPT
_need = []
for f in FOLDS:
    _tr, _va = run_ids(f)
    _need += _tr + _va
_need = list(dict.fromkeys(_need + gold_ids))
EMB, EMB_TAG, EMB_S = embed_windows(CKPT, _need)
EMB = EMB.to(DEVICE)
EROW = {u: i for i, u in enumerate(_need)}
F_EMB = int(EMB.shape[-1])
assert torch.isfinite(EMB.float()).all(), "embeddings con NaN/inf"


@torch.no_grad()
def predict(head, ids):
    head.eval()
    if not ids:
        return pd.DataFrame(columns=LABELS)
    out = []
    for i in range(0, len(ids), 256):
        r = torch.tensor([EROW[u] for u in ids[i:i + 256]], device=DEVICE)
        out.append(torch.sigmoid(head(EMB[r].float())).cpu().numpy())
    return pd.DataFrame(np.concatenate(out), index=pd.Index(ids, name="StudyInstanceUID"), columns=LABELS)


def train_head(fold, kind, seed=SEED, save=True):
    """Bucle del padre (pérdida, pos_weight, AdamW + OneCycle, clip, selección y top-K SWA por pseudo L2) con la cabeza
    `kind` sobre embeddings congelados; batch de BATCH_SIZE estudios × K_SEQ ventanas ordenadas."""
    seed_everything(seed + fold)
    tr_ids, va_ids = run_ids(fold)
    sfx = f"{kind}_{TAG}{'' if ALL_DATA else fold}" + ("" if seed == SEED else f"_s{seed}")
    head = make_head(kind, F_EMB)
    prev = np.clip(np.nanmean(soft.loc[tr_ids, LABELS].to_numpy(), 0), 0.03, 0.7)
    lossf = nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(np.clip((1 - prev) / prev, 1, 10), dtype=torch.float32, device=DEVICE))
    Ytr = torch.tensor(soft.loc[tr_ids, LABELS].to_numpy(), dtype=torch.float32, device=DEVICE)
    rtr = torch.tensor([EROW[u] for u in tr_ids], device=DEVICE)
    steps = len(tr_ids) // BATCH_SIZE                       # drop_last, como el loader del padre
    opt = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=HEAD_LR, total_steps=max(1, steps * EPOCHS), pct_start=PCT_START)
    gen = torch.Generator().manual_seed(seed + fold)
    n_par = sum(p.numel() for p in head.parameters())
    print(f"[{sfx}] cabeza {kind} ({n_par / 1e6:.2f} M parámetros) · train {len(tr_ids)} · val {len(va_ids)} · {steps} pasos/época", flush=True)
    history, topk, best, keep = [], [], None, {}
    start = time.time()
    for epoch in range(EPOCHS):
        head.train(); tot = 0.0; t_ep = time.time()
        perm = torch.randperm(len(tr_ids), generator=gen).to(DEVICE)
        for step in range(steps):
            b = perm[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
            y = Ytr[b]
            opt.zero_grad(set_to_none=True)
            logits = head(EMB[rtr[b]].float())
            m = ~torch.isnan(y)
            loss = (lossf(logits, torch.nan_to_num(y)) * m).sum() / m.sum().clamp(min=1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP)
            opt.step(); sched.step(); tot += loss.item()
        t_train = time.time() - t_ep
        pv, pg = predict(head, va_ids), predict(head, gold_ids)
        sv, sg = score(pv), score(pg)
        rec = {"epoch": epoch, "train_loss": tot / max(1, steps),
               "pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None,
               "v1_auc": sv["v1"]["mean"] if sv["v1"] else None,
               "gold_auc": sg["gold"]["mean"], "train_s": t_train, "eval_s": time.time() - t_ep - t_train,
               "minutes": (time.time() - start) / 60}
        history.append(rec)
        if save:
            pd.DataFrame(history).to_csv(RESULTS / f"history_{sfx}.csv", index=False)
        print(f"[{sfx}] ep{epoch} loss {rec['train_loss']:.4f} | pseudo L2 {rec['pseudo_auc'] or float('nan'):.4f} | "
              f"v1 {rec['v1_auc'] or float('nan'):.4f} | gold {rec['gold_auc']:.4f} (solo veto) | {t_train:.1f}s", flush=True)
        state = {k_: v.detach().cpu().clone() for k_, v in head.state_dict().items()}
        if ALL_DATA:
            if epoch in SWA_EPOCHS:
                keep[epoch] = state
            continue
        crit = rec["pseudo_auc"]
        if best is None or crit > best["pseudo_auc"]:
            best = dict(rec)
            if save:
                torch.save({"head": state, "kind": kind, "F": F_EMB, "gru_hidden": GRU_HIDDEN, "k_seq": K_SEQ, "epoch": epoch,
                            "exp": EXP, "pseudo_auc": crit, "backbone_ckpt": CKPT}, RESULTS / f"best_{sfx}.pt")
                save_preds(pv, pg, f"best_{sfx}")
            best["pseudo_per_label"], best["gold_per_label"] = sv["pseudo"]["per_label"], sg["gold"]["per_label"]
        topk.append((crit, epoch, state)); topk.sort(key=lambda t: -t[0]); del topk[TOPK:]
        del state
    gpu_h = (time.time() - start) / 3600
    if ALL_DATA:
        eps = sorted(keep) or [EPOCHS - 1]
        states = [keep[e] for e in sorted(keep)] or [head.state_dict()]
    else:
        eps = [e for _, e, _ in topk]
        states = [s for _, _, s in topk]
    avg = average_states(states)
    head.load_state_dict(avg)
    pv, pg = predict(head, va_ids), predict(head, gold_ids)
    sv, sg = score(pv), score(pg)
    if save:
        torch.save({"head": avg, "kind": kind, "F": F_EMB, "gru_hidden": GRU_HIDDEN, "k_seq": K_SEQ, "swa_over": eps,
                    "exp": EXP, "backbone_ckpt": CKPT}, RESULTS / f"swa_{sfx}.pt")
        save_preds(pv, pg, f"swa_{sfx}")
        if ALL_DATA:   # checkpoint completo para el kernel: backbone all-data congelado + cabeza SWA
            _bb = {k_: v for k_, v in torch.load(CKPT, map_location="cpu", weights_only=False)["model"].items() if k_.startswith("backbone.")}
            torch.save({"model": {**_bb, **avg}, "arch": ARCH, "res": RES_IN, "lab": LABELS, "head": kind,
                        "gru_hidden": GRU_HIDDEN, "k_seq": K_SEQ, "swa_over": eps, "exp": EXP, "backbone_ckpt": CKPT},
                       RESULTS / f"full_{sfx}.pt")
            del _bb
    swa = {"epochs": eps, "pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None,
           "v1_auc": sv["v1"]["mean"] if sv["v1"] else None, "gold_auc": sg["gold"]["mean"],
           "pseudo_per_label": sv["pseudo"]["per_label"] if sv["pseudo"] else None, "gold_per_label": sg["gold"]["per_label"]}
    print(f"[{sfx}] SWA sobre épocas {eps} | pseudo L2 {swa['pseudo_auc'] or float('nan'):.4f} | "
          f"v1 {swa['v1_auc'] or float('nan'):.4f} | gold {swa['gold_auc']:.4f}", flush=True)
    out = {"head": kind, "seed": seed, "n_params": n_par, "n_train": len(tr_ids), "n_val": len(va_ids), "history": history,
           "swa": swa, "gpu_hours": gpu_h, "swa_ckpt_path": str(RESULTS / f"swa_{sfx}.pt") if save else None}
    if best is not None:
        out.update({"pseudo_auc": best["pseudo_auc"], "v1_auc": best["v1_auc"], "gold_auc": best["gold_auc"],
                    "best_epoch": best["epoch"], "pseudo_per_label": best["pseudo_per_label"],
                    "gold_per_label": best["gold_per_label"], "ckpt_path": str(RESULTS / f"best_{sfx}.pt") if save else None})
    if ALL_DATA and save:
        out["full_ckpt_path"] = str(RESULTS / f"full_{sfx}.pt")
    return out


PARAMS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}

# Informativo: la cabeza del padre SIN reentrenar sobre estos embeddings (= padre con K_SEQ ventanas, como en el kernel)
PARENT_K64 = None
if not ALL_DATA:
    _ph = make_head("mil", F_EMB)
    _sd = torch.load(CKPT, map_location="cpu", weights_only=False)["model"]
    _ph.load_state_dict({k_: v for k_, v in _sd.items() if not k_.startswith("backbone.")}, strict=True)
    del _sd
    _sv, _sg = score(predict(_ph, run_ids(FOLDS[0])[1])), score(predict(_ph, gold_ids))
    PARENT_K64 = {"pseudo_auc": _sv["pseudo"]["mean"] if _sv["pseudo"] else None, "gold_auc": _sg["gold"]["mean"]}
    print(f"padre sin reentrenar con {K_SEQ} ventanas: pseudo L2 {PARENT_K64['pseudo_auc'] or float('nan'):.4f} | "
          f"gold {PARENT_K64['gold_auc']:.4f}", flush=True)

per_fold, control, info_seeds = {}, {}, {}
for fold in FOLDS:
    key = "alldata" if ALL_DATA else str(fold)
    per_fold[key] = train_head(fold, HEAD)
    if not ALL_DATA:
        control[key] = train_head(fold, "mil")                 # control obligatorio: cabeza del padre reentrenada
        if not SMOKE:                                          # semillas extra: solo informativas, no deciden
            info_seeds[key] = {}
            for k in ("mil", HEAD):
                info_seeds[key][k] = {}
                for s in INFO_SEEDS:
                    _sw = train_head(fold, k, seed=s, save=False)["swa"]
                    info_seeds[key][k][str(s)] = {"pseudo_auc": _sw["pseudo_auc"], "gold_auc": _sw["gold_auc"]}
GPU_H_TOTAL = (time.time() - T_CELL4) / 3600
print(f"celda 4: {GPU_H_TOTAL:.3f} GPU-h (embeddings {EMB_S:.0f} s)")

## 5. Evaluación (pseudo L2 decide; v1 y gold informativos/veto)

In [ ]:
rows = {}
for name, res in ((HEAD, per_fold), ("control mil", control)):
    for f, r in res.items():
        rows[f"{name} {f} mejor época ({r.get('best_epoch')})"] = {"pseudo L2": r.get("pseudo_auc"), "v1": r.get("v1_auc"), "gold (veto)": r.get("gold_auc")}
        rows[f"{name} {f} SWA {r['swa']['epochs']}"] = {"pseudo L2": r["swa"]["pseudo_auc"], "v1": r["swa"]["v1_auc"], "gold (veto)": r["swa"]["gold_auc"]}
summary = pd.DataFrame(rows).T
display(summary.round(4))
parent = (leer_metricas(PADRE) if PADRE else None) or {}
r0, c0 = per_fold.get("0", {}), control.get("0", {})
P_SWA = parent.get("swa_f0") or {}
swa0, cswa0 = r0.get("swa") or {}, c0.get("swa") or {}
_d = lambda a, b: a - b if a is not None and b is not None else None
delta_pseudo = _d(swa0.get("pseudo_auc"), P_SWA.get("pseudo_auc"))
delta_gold = _d(swa0.get("gold_auc"), P_SWA.get("gold_auc"))
delta_vs_control = _d(swa0.get("pseudo_auc"), cswa0.get("pseudo_auc"))
DECISION = decision(delta_pseudo, delta_gold)   # regla genérica del equipo (informativa; decide la SPEC)
if swa0.get("pseudo_auc") is None:
    VERDICT_AUTO = None
elif swa0["gold_auc"] < CRIT_KILLED:
    VERDICT_AUTO = "killed"
elif swa0["pseudo_auc"] >= CRIT_WORKED and cswa0.get("pseudo_auc") is not None and swa0["pseudo_auc"] > cswa0["pseudo_auc"]:
    VERDICT_AUTO = "worked"
else:
    VERDICT_AUTO = "didnt"
print(f"padre {PADRE} SWA f0: pseudo {P_SWA.get('pseudo_auc')} · gold {P_SWA.get('gold_auc')} | padre sin reentrenar K{K_SEQ}: {PARENT_K64}")
print(f"{HEAD} SWA f0: pseudo {swa0.get('pseudo_auc')} · gold {swa0.get('gold_auc')} | control SWA f0: pseudo {cswa0.get('pseudo_auc')} · gold {cswa0.get('gold_auc')}")
print(f"Δ pseudo vs padre {delta_pseudo} · Δ gold vs padre {delta_gold} · Δ pseudo vs control {delta_vs_control}")
print(f"Semillas informativas: {info_seeds}")
print("Decisión genérica:", DECISION, "| VERDICT_AUTO (criterios SPEC):", VERDICT_AUTO)
for name, res in ((HEAD, per_fold), ("control mil", control)):
    for f, r in res.items():
        print(name, f)
        display(pd.DataFrame(r["history"]).set_index("epoch").round(4))


# seqhead: coste de inferencia de la cabeza (fp16, 1 estudio de K_SEQ ventanas, como el kernel). El backbone no cambia.
@torch.no_grad()
def bench_head(kind, n=300):
    h = make_head(kind, F_EMB).half().eval()
    x = torch.randn(1, K_SEQ, F_EMB, device=DEVICE).half()
    for _ in range(20):
        h(x)
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(n):
        h(x)
    torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000


BENCH = {"head_ms_per_study": {k: bench_head(k) for k in dict.fromkeys(("mil", HEAD))},
         "backbone_s_per_study_parent": (parent.get("bench_inference") or {}).get("exp057_cfg", {}).get("s_per_study"),
         "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu"}
print(json.dumps(BENCH, indent=1))

## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else ("alldata" if ALL_DATA else "screen"),
    "select_on": "pseudo_L2_bin>0.5", "params": PARAMS, "per_fold": per_fold,
    "gold_auc": r0.get("gold_auc"), "gold_per_label": r0.get("gold_per_label"), "oof_pseudo_auc": None,
    "swa_f0": r0.get("swa"),
    "head": HEAD, "control": control, "control_swa_f0": c0.get("swa"),                   # seqhead
    "parent_swa_f0": {"pseudo_auc": P_SWA.get("pseudo_auc"), "gold_auc": P_SWA.get("gold_auc")},
    "parent_k64": PARENT_K64, "info_seeds": info_seeds,
    "delta": {"pseudo_f0": delta_pseudo, "gold": delta_gold, "pseudo_f0_vs_control": delta_vs_control},
    "decision_auto": DECISION, "verdict_auto": VERDICT_AUTO,
    "criteria": {"worked": f"SWA pseudo f0 >= {CRIT_WORKED} and > control SWA pseudo f0",
                 "killed": f"SWA gold f0 < {CRIT_KILLED}"},
    "backbone_ckpt": CKPT,
    "embeddings": {"tag": EMB_TAG, "seconds": EMB_S, "n_studies": len(_need), "k_seq": K_SEQ, "F": F_EMB},
    "corpus": {"path": CORPUS, "n": len(corpus_ids), "D": D, "img": IMG, "build_seconds": meta.get("seconds"),
               "n_errors": len(meta["errors"])},
    "gpu_hours_total": GPU_H_TOTAL,
    "bench_inference": BENCH,
}
# all-data no pisa el metrics.json del fold 0: va a results/<EXP>/alldata/metrics.json
print("Guardado:", guardar_metricas(RESULTS / "alldata" if ALL_DATA else RESULTS, metrics))

## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza la fila en `docs/EXPERIMENTOS.md`
(estado `worked`, `didnt` o `killed`). Después: commit y PR a `main`.